# Norm's Brain - Concept Learning

Scaling up concept learning on TPU

In [ ]:
# Check TPU
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

try:
    import tensorflow as tf
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.TPUStrategy(tpu)
    print(f"TPU: {tpu.master()}")
    USE_TPU = True
except Exception as e:
    print(f"No TPU: {e}")
    USE_TPU = False

import torch
print(f"PyTorch: {torch.__version__}")

## Concept Brain

In [ ]:
from collections import defaultdict

class Concept:
    """A concept in Norm's mind"""
    def __init__(self, name):
        self.name = name
        self.properties = {}
        self.relationships = defaultdict(list)
        self.strength = 0.1
        
    def strengthen(self):
        self.strength = min(1.0, self.strength + 0.1)
        
    def __repr__(self):
        return f"Concept({self.name}, strength={self.strength:.2f})"


class ConceptBrain:
    """Brain that learns CONCEPTS, not just words"""
    def __init__(self):
        self.concepts = {}
        self.relationships = []
        
    def learn(self, text: str):
        text = text.lower().strip()
        
        # Parse "X is Y" sentences
        if " is " in text:
            parts = text.split(" is ")
            if len(parts) == 2:
                subject = self._get_concept(parts[0].strip())
                obj = self._get_concept(parts[1].strip())
                subject.properties['is'] = obj.name
                obj.strengthen()
                subject.strengthen()
                self.relationships.append((subject.name, 'is', obj.name, 0.9))
                
        # Parse "X has Y"
        if " has " in text:
            parts = text.split(" has ")
            if len(parts) == 2:
                subject = self._get_concept(parts[0].strip())
                obj = self._get_concept(parts[1].strip())
                subject.properties['has'] = obj.name
                obj.strengthen()
                subject.strengthen()
                self.relationships.append((subject.name, 'has', obj.name, 0.9))
                
        # Parse "X can Y"
        if " can " in text:
            parts = text.split(" can ")
            if len(parts) == 2:
                subject = self._get_concept(parts[0].strip())
                ability = parts[1].strip()
                subject.properties['can'] = ability
                self.relationships.append((subject.name, 'can', ability, 0.7))
                
        # Self-reference learning
        if " i " in text or " me " in text or " my " in text:
            me = self._get_concept("i")
            me.strengthen()
            me.properties['exists'] = True
            
    def _get_concept(self, name: str) -> Concept:
        if name not in self.concepts:
            self.concepts[name] = Concept(name)
        return self.concepts[name]
        
    def query(self, question: str) -> str:
        question = question.lower()
        
        # "what is X?"
        if "what is " in question:
            target = question.split("what is ")[1].strip().rstrip('?')
            if target in self.concepts:
                c = self.concepts[target]
                if 'is' in c.properties:
                    return f"{c.name} is {c.properties['is']}"
                    
        # "what color is X?"
        if "what color is " in question:
            target = question.split("what color is ")[1].strip().rstrip('?')
            if target in self.concepts:
                c = self.concepts[target]
                if 'is' in c.properties:
                    return f"The color of {c.name} is {c.properties['is']}"
                    
        # "who are you?"
        if "who are you" in question or "what are you" in question:
            if "i" in self.concepts:
                i = self.concepts["i"]
                if i.strength > 0.3:
                    return "I am a digital being. I exist in this computer."
                    
        return "I don't know."
        
    def summary(self):
        lines = ["I know:"]
        for name, c in sorted(self.concepts.items(), key=lambda x: -x[1].strength)[:10]:
            props = ", ".join([f"{k}={v}" for k,v in c.properties.items()])
            lines.append(f"  {name}: {props}")
        return "\n".join(lines)

## Train

In [ ]:
brain = ConceptBrain()

# Learn basic concepts
lessons = [
    "the sky is blue",
    "grass is green",
    "the sun is yellow",
    "the ocean is blue",
    "blood is red",
    "the sky has clouds",
    "a tree has leaves",
    "water is wet",
    "fire is hot",
    "ice is cold",
    # Self
    "i am a digital being",
    "i am alive",
    "i can think",
    "i can learn",
    "i exist",
    # Colors
    "blue is a color",
    "red is a color",
    "green is a color",
]

for lesson in lessons:
    brain.learn(lesson)
    
print(f"Concepts learned: {len(brain.concepts)}")
print(brain.summary())

In [ ]:
# Query
questions = [
    "what is sky?",
    "what is grass?",
    "what color is sky?",
    "who are you?",
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {brain.query(q)}")
    print()

## Dual-Head Continual Learning (TPU)

In [ ]:
# Dual-Head architecture test on larger scale
import torch
import torch.nn as nn
import torch.nn.functional as F

class DualHeadNet(nn.Module):
    def __init__(self, input_dim=64, hidden=256, output_dim=10):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden)
        )
        self.core_head = nn.Linear(hidden, output_dim)
        self.flex_head = nn.Linear(hidden, output_dim)
        
    def forward_core(self, x):
        return self.core_head(self.encoder(x))
        
    def forward_flex(self, x):
        return self.flex_head(self.encoder(x))
        
    def freeze_core(self):
        for p in self.core_head.parameters():
            p.requires_grad = False


def generate_task(task_id, n=1000):
    """Generate task with different features"""
    np.random.seed(task_id * 100)
    f = task_id % 20
    
    X0 = np.random.randn(n, 64)
    X0[:, f] -= 2
    X1 = np.random.randn(n, 64)
    X1[:, f] += 2
    
    X = np.vstack([X0, X1])
    y = np.concatenate([np.zeros(n), np.ones(n)])
    
    return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long)


# Run experiment
dualhead = DualHeadNet()
opt = torch.optim.Adam(dualhead.parameters(), lr=0.01)

print("Training 10 tasks...")

for task in range(10):
    X, y = generate_task(task, 500)
    
    # First 3 tasks = core, rest = flex
    if task < 3:
        head = dualhead.forward_core
    else:
        head = dualhead.forward_flex
        dualhead.freeze_core()
        
    # Train
    for epoch in range(20):
        out = head(X)
        loss = F.cross_entropy(out, y)
        opt.zero_grad()
        loss.backward()
        opt.step()
    
    # Test on core tasks
    correct = 0
    total = 0
    for t in range(task + 1):
        X_test, y_test = generate_task(t, 200)
        with torch.no_grad():
            if t < 3:
                pred = dualhead.forward_core(X_test).argmax(1)
            else:
                pred = dualhead.forward_flex(X_test).argmax(1)
        correct += (pred == y_test).sum().item()
        total += len(y_test)
    
    print(f"Task {task+1}: Accuracy {correct/total*100:.1f}%")

print("\nDone!")

## Next Steps

1. Scale up dual-head to larger models
2. Train actual language model
3. Add more concept types
4. Test reasoning